# 01 - Dataset Import

In [ ]:
from itertools import combinations
import numpy as np
import os
import pandas as pd
from PIL import Image
import shutil

import sys
sys.path.append('.')
from config import PROJECT_DIR, KAGGLE_DIR, DATASET_DIR, DATASET_ZIP, PHYSIONET_DIR
from utils import parse_dicom_age, age_to_group

This notebook downloads the VinDr-CXR dataset from Kaggle, applies the filtering pipeline and generates the final dataset with images at 1024×1024 and 256×256.

Kaggle API credentials are required to download the dataset. You can either place your `kaggle.json` file in `~/.kaggle/` or uncomment and fill in the credentials directly in the cell below.

The N_HEALTHY constant adds to the filtered dataset n 'no finding' radiographies.

Consensus is unanimous (all 3 assigned radiologists must mark a finding). The IoU spatial-agreement threshold is selected separately per pathology (aortic enlargement, cardiomegaly): each threshold is auto-picked, from a sweep of candidate values, as the value just above the steepest single-step drop in retained-image count.

In [ ]:
# Set Kaggle API credentials to be able to download the dataset
# os.environ['KAGGLE_USERNAME'] = ""
# os.environ['KAGGLE_KEY'] = ""

N_HEALTHY = 4000
# Per-pathology IoU thresholds are auto-selected in Step 2 below, from the
# retention-vs-threshold curve. These are fallback values only, used if a
# pathology's curve has no clear steep drop.
IOU_THRESHOLD_AORTIC = 0.4
IOU_THRESHOLD_CARDIOMEGALY = 0.4

In [ ]:
# Dataset xhlulu kaggle (images 1024x1024 PNG)
!kaggle datasets download -d xhlulu/vinbigdata-chest-xray-resized-png-1024x1024
!unzip -q vinbigdata-chest-xray-resized-png-1024x1024.zip -d {KAGGLE_DIR}/
!rm vinbigdata-chest-xray-resized-png-1024x1024.zip

# Official dataset VinDr-CXR (file: train.csv)
!kaggle competitions download -c vinbigdata-chest-xray-abnormalities-detection -f train.csv
!mv train.csv {KAGGLE_DIR}/train.csv 

print("Dataset VinDr-CXR 1024x1024 PNG downloaded and ready for processing.")

In [ ]:
print(f"Total of downloaded images: {len(os.listdir(f'{KAGGLE_DIR}/train'))}")

## Patient demographics (PatientAge, PatientSex) from PhysioNet

Downloads the official VinDr-CXR DICOM metadata from PhysioNet (requires the
PhysioNet credentialing/verification course). Separate from Kaggle's
`train.csv` (pathology labels) - provides per-image `PatientAge` and
`PatientSex`, used later for stratified splitting and demographic
stratification of all metrics.

In [ ]:
# PhysioNet credentials required (same account as the VinDr-CXR verification course).
# os.environ['PHYSIONET_USER'] = ""
# os.environ['PHYSIONET_PASS'] = ""

os.makedirs(PHYSIONET_DIR, exist_ok=True)
physionet_user = os.environ.get('PHYSIONET_USER', '')
physionet_pass = os.environ.get('PHYSIONET_PASS', '')

# NOTE: filename/URL for the DICOM tags export is inferred from PhysioNet's
# vindr-cxr 1.0.0 layout, not confirmed against a live download. Verify before running.
!wget -q --user {physionet_user} --password {physionet_pass} -O {PHYSIONET_DIR}/dicom_tags_train.csv https://physionet.org/files/vindr-cxr/1.0.0/dicom_tags/dicom_tags_train.csv

print("VinDr-CXR DICOM demographic metadata downloaded.")

In [ ]:
df_demographics = pd.read_csv(f'{PHYSIONET_DIR}/dicom_tags_train.csv')
df_demographics = df_demographics[['image_id', 'PatientAge', 'PatientSex']].copy()
df_demographics['PatientAge'] = df_demographics['PatientAge'].apply(parse_dicom_age)
df_demographics['PatientAgeGroup'] = df_demographics['PatientAge'].apply(age_to_group)

n_missing = df_demographics['PatientAge'].isna().sum()
print(f"Demographics loaded for {len(df_demographics)} images.")
print(f"Unparseable/missing PatientAge: {n_missing}")

## Reescale coordinates and define needed functions


In [ ]:
# Load labels (official) y metadata (original dimensions to reescale coordenates)
df_labels = pd.read_csv(f'{KAGGLE_DIR}/train.csv')
df_meta = pd.read_csv(f'{KAGGLE_DIR}/train_meta.csv') # This one has 'width' and 'height' original dimensions

# We merge using the correct names of the dataset of xhlulu (dim1=width, dim0=height)
df_master = pd.merge(df_labels, df_meta[['image_id', 'dim0', 'dim1']], on='image_id')

# ¿are they already scaled?
if df_master['x_max'].max() > 1024:
    print("Original coordinates detected. Rescaling to 1024x1024...")

    df_master['x_min'] = df_master['x_min'] * (1024 / df_master['dim1'])
    df_master['x_max'] = df_master['x_max'] * (1024 / df_master['dim1'])
    df_master['y_min'] = df_master['y_min'] * (1024 / df_master['dim0'])
    df_master['y_max'] = df_master['y_max'] * (1024 / df_master['dim0'])

    print("Coordinates scaled successfully.")
else:
    print("Coordinates already scaled. No action needed.")

In [ ]:
def calculate_iou(box1, box2):
    """
    box: (x_min, y_min, x_max, y_max)
    """
    x_min = max(box1[0], box2[0])
    y_min = max(box1[1], box2[1])
    x_max = min(box1[2], box2[2])
    y_max = min(box1[3], box2[3])

    intersection = max(0, x_max - x_min) * max(0, y_max - y_min)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection

    return intersection / union if union > 0 else 0.0

# IoU mínimo por pares entre radiólogos
def minimum_iou_between_pairs(grupo):
    """
    Given a group of rows (annotations from different radiologists on the same image and class), 
    calculate the minimum IoU between all possible pairs.
    If there is only one radiologist with consensus, return 1.0.

    """
    boxes = grupo[['x_min', 'y_min', 'x_max', 'y_max']].values
    if len(boxes) < 2:
        return 1.0
    ious = [calculate_iou(b1, b2) for b1, b2 in combinations(boxes, 2)]
    return min(ious)

## Filter dataset pipeline

In [ ]:
# Step 0: initial state dataset
print("=" * 60)
print("Step 0 - Initial state of the dataset")
print("=" * 60)
clases_tfg = [0, 3, 14]
df_interest = df_master[df_master['class_id'].isin(clases_tfg)].copy()
print(f"Total rows (annotations): {len(df_interest)}")
print(f"Unique images:             {df_interest['image_id'].nunique()}")
print()

# Step 1: Consensus filtering (≥ 2 different radiologists)
print("=" * 60)
print("Step 1 - Consensus filtering (≥ 2 different radiologists)")
print("=" * 60)

votes = (df_interest
         .groupby(['image_id', 'class_id'])['rad_id']
         .nunique()
         .reset_index()
         .rename(columns={'rad_id': 'n_votes'}))

# Unanimous consensus: all 3 assigned radiologists must agree on the finding
consensus_ids = votes[votes['n_votes'] == 3][['image_id', 'class_id']]
df_consensus = pd.merge(df_interest, consensus_ids, on=['image_id', 'class_id'])

# Only pathological classes (aneurysm and cardiomegaly) for the next steps, we will add healthy at the end
df_pathological = df_consensus[df_consensus['class_id'] != 14]

print(f"Images with semantic consensus (pathological): {df_pathological['image_id'].nunique()}")
print(df_pathological.groupby('class_name')['image_id'].nunique().to_string())
print()

# Step 2: Spatial filter (minimum IoU per pair, per-pathology threshold)
print("=" * 60)
print("Step 2 — Spatial filter (per-pathology IoU threshold)")
print("=" * 60)

iou_por_grupo = (df_pathological
                 .groupby(['image_id', 'class_id'])
                 .apply(minimum_iou_between_pairs)
                 .reset_index()
                 .rename(columns={0: 'iou_min'}))

# Auto-select each pathology's threshold as the value just above the
# steepest single-step drop in retained-image count across an IoU sweep.
def select_iou_threshold(iou_series, fallback, sweep=np.arange(0.05, 0.95, 0.05)):
    retention = [int((iou_series >= t).sum()) for t in sweep]
    drops = [retention[i] - retention[i + 1] for i in range(len(retention) - 1)]
    if not drops or max(drops) <= 0:
        return fallback, list(zip(sweep.round(2), retention))
    steepest = int(np.argmax(drops))
    return round(float(sweep[steepest]), 2), list(zip(sweep.round(2), retention))

thresholds = {}
for class_id, class_label, fallback in [
    (0, 'Aortic enlargement', IOU_THRESHOLD_AORTIC),
    (3, 'Cardiomegaly', IOU_THRESHOLD_CARDIOMEGALY),
]:
    class_ious = iou_por_grupo[iou_por_grupo['class_id'] == class_id]['iou_min']
    t, curve = select_iou_threshold(class_ious, fallback)
    thresholds[class_id] = t
    print(f"{class_label}: retention curve (threshold: n_images_retained)")
    for thr, n in curve:
        print(f"    {thr:.2f}: {n}")
    print(f"  -> auto-selected threshold: {t}\n")

IOU_THRESHOLD_AORTIC = thresholds[0]
IOU_THRESHOLD_CARDIOMEGALY = thresholds[3]

def iou_threshold_for(class_id):
    return IOU_THRESHOLD_AORTIC if class_id == 0 else IOU_THRESHOLD_CARDIOMEGALY

iou_por_grupo['threshold'] = iou_por_grupo['class_id'].apply(iou_threshold_for)
aproved = iou_por_grupo[iou_por_grupo['iou_min'] >= iou_por_grupo['threshold']][['image_id', 'class_id']]
discarded_iou = iou_por_grupo[iou_por_grupo['iou_min'] < iou_por_grupo['threshold']][['image_id', 'class_id']]

print(f"Aortic enlargement IoU threshold: {IOU_THRESHOLD_AORTIC}")
print(f"Cardiomegaly IoU threshold:       {IOU_THRESHOLD_CARDIOMEGALY}")
print(f"Images discarded due to low spatial agreement: {len(discarded_iou)}")
print(f"  → Aortic enlargement discarded:    {len(discarded_iou[discarded_iou['class_id'] == 0])}")
print(f"  → Cardiomegaly discarded:{len(discarded_iou[discarded_iou['class_id'] == 3])}")
print()

df_aproved = pd.merge(df_pathological, aproved, on=['image_id', 'class_id'])
print(f"Images that pass the spatial filter: {df_aproved['image_id'].nunique()}")
print(df_aproved.groupby('class_name')['image_id'].nunique().to_string())
print()

# Step 3: Bounding Box Union (for each image-class, unify into a single BB that encompasses all radiologists)
print("=" * 60)
print("Step 3 — Bounding Box Union")
print("=" * 60)

final_BB = (df_aproved
            .groupby(['image_id', 'class_id', 'class_name'])
            .agg(x_min=('x_min', 'min'),
                 y_min=('y_min', 'min'),
                 x_max=('x_max', 'max'),
                 y_max=('y_max', 'max'))
            .reset_index())

print(f"Final BBs generated: {len(final_BB)}")
print(final_BB.groupby('class_name')['image_id'].nunique().to_string())
print()

# Step 4: Control group (healthy)
print("=" * 60)
print("Step 4 — Control group (healthy)")
print("=" * 60)

# Only images where all radiologists said "no finding" (class_id=14)
ids_healthy = (df_master.groupby('image_id')['class_id']
                  .apply(lambda x: (x == 14).all())
                  .reset_index()
                  .query('class_id == True')['image_id'])

df_healthy = (df_master[df_master['image_id'].isin(ids_healthy)]
            .drop_duplicates('image_id')
            .sample(n=N_HEALTHY, random_state=42))

print(f"Healthy images selected: {len(df_healthy)}")
print()

# Step 5: Final dataset
print("=" * 60)
print("Step 5 — Final dataset")
print("=" * 60)

df_1024 = pd.concat([final_BB, df_healthy[['image_id', 'class_id', 'class_name',
                                               'x_min', 'y_min', 'x_max', 'y_max']]])

# Sequential ID mapping
mapping_ids = {
    0: 0,
    3: 1,
    14: 2
}
#If you need to map the class names to another language uncomment this part and adapt the names as you wish
#mapping_names = {
#    'Aortic enlargement': 'aneurisma dell'aorta',
#    'Cardiomegaly':       'cardiomegalia',
#    'No finding':         'nessun risultato'
#}
df_1024['class_id'] = df_1024['class_id'].map(mapping_ids)
#df_1024['class_name'] = df_1024['class_name'].map(mapping_names)

# Images with both pathologies
ids_aneurysm     = set(df_1024[df_1024['class_id'] == 0]['image_id'])
ids_cardiomegaly = set(df_1024[df_1024['class_id'] == 1]['image_id'])
both = ids_aneurysm & ids_cardiomegaly

print(f"Aortic aneurysm:      {len(ids_aneurysm)}")
print(f"Cardiomegaly:  {len(ids_cardiomegaly)}")
print(f"Healthy:          {len(df_healthy)}")
print(f"Both:          {len(both)}")
print(f"Total unique images: {df_1024['image_id'].nunique()}")
print("=" * 60)

## Compress Dataset

In [ ]:
os.makedirs(f'{DATASET_DIR}/images_1024', exist_ok=True)
os.makedirs(f'{DATASET_DIR}/images_256', exist_ok=True)

print("Processing images...")

for img_id in df_1024['image_id'].unique():
    src = f'{KAGGLE_DIR}/train/{img_id}.png'
    if os.path.exists(src):
        # Save 1024x1024 version
        shutil.copy(src, f'{DATASET_DIR}/images_1024/{img_id}.png')

        # Generate and save the 256x256 version (Bicubic)
        img = Image.open(src)
        img_256 = img.resize((256, 256), resample=Image.BICUBIC)
        img_256.save(f'{DATASET_DIR}/images_256/{img_id}.png')

# Cross-reference with patient demographics (section 1.1)
df_1024 = pd.merge(df_1024, df_demographics, on='image_id', how='left')
n_unmatched = df_1024['PatientAge'].isna().sum()
if n_unmatched:
    print(f"Warning: {n_unmatched} image(s) have no demographic match.")
df_1024 = df_1024.rename(columns={
    'PatientAge': 'patient_age',
    'PatientSex': 'patient_sex',
    'PatientAgeGroup': 'patient_age_group'
})

# Save the metadata within the same folder
df_1024.to_csv(f'{DATASET_DIR}/metadata_1024.csv', index=False)

# Create the CSV for 256x256, scaling the coordinates (factor 0.25)
df_256 = df_1024.copy()
for col in ['x_min', 'y_min', 'x_max', 'y_max']:
    df_256[col] = df_256[col] * 0.25
df_256.to_csv(f'{DATASET_DIR}/metadata_256.csv', index=False)

# Verify BB coordinate scaling (section 1.3)
scale_check = df_1024.set_index(['image_id', 'class_id'])[['x_min', 'y_min', 'x_max', 'y_max']] * 0.25
scale_actual = df_256.set_index(['image_id', 'class_id'])[['x_min', 'y_min', 'x_max', 'y_max']]
scale_check, scale_actual = scale_check.align(scale_actual, join='inner')
max_diff = (scale_check - scale_actual).abs().max().max()
print(f"Max BB scaling discrepancy (256 vs 1024*0.25): {max_diff:.6f} px")
assert max_diff < 1e-6, "Bounding box scaling factor 0.25 not applied correctly!"
print("Bounding box scaling verified correct.")

## Restore

In [ ]:
os.makedirs(PROJECT_DIR, exist_ok=True)
!zip -rq {DATASET_ZIP} {DATASET_DIR}
shutil.rmtree(KAGGLE_DIR)
shutil.rmtree(DATASET_DIR) 

print("Dataset processed and ready for training.")
print('01_dataset_import completed.')

# 02 - Preprocessing, Images and Training

In [ ]:
from codecarbon import EmissionsTracker
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import shutil
from sklearn.metrics import matthews_corrcoef
from sklearn.model_selection import train_test_split
import time
import torch
from torch.amp import autocast, GradScaler
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms

import sys
sys.path.append('.')
from config import PROJECT_DIR, DATASET_DIR, DATASET_ZIP, MODELS_DIR
from utils import apply_nclahe, ChestXrayDataset

## Configuration

This notebook trains AlexNet and DenseNet-121 on chest X-ray radiographs from the VinDr-CXR dataset at two resolutions (256×256 and 1024×1024) for multi-label classification of aortic enlargement and cardiomegaly.

In [ ]:
!unzip -q {DATASET_ZIP} -d {PROJECT_DIR}

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
df = pd.read_csv(f'{DATASET_DIR}/metadata_1024.csv')

In [ ]:
# Classify each image into a category for the split
def category_split(image_id, ids_aneurysm, ids_cardiomegaly):
    in_aneurysm = image_id in ids_aneurysm
    in_cardio = image_id in ids_cardiomegaly
    if in_aneurysm and in_cardio:
        return 'both'
    elif in_aneurysm:
        return 'aortic enlargement'
    elif in_cardio:
        return 'cardiomegaly'
    else:
        return 'healthy'

ids_aneurysm = set(df[df['class_id'] == 0]['image_id'])
ids_cardiomegaly = set(df[df['class_id'] == 1]['image_id'])

df_unique = pd.DataFrame({'image_id': df['image_id'].unique()})
df_unique['category'] = df_unique['image_id'].apply(
    lambda x: category_split(x, ids_aneurysm, ids_cardiomegaly)
)

# Patient demographics (added to metadata in 01_dataset_import, section 1.1)
df_demo = df.drop_duplicates('image_id')[['image_id', 'patient_sex', 'patient_age_group']]
df_unique = pd.merge(df_unique, df_demo, on='image_id', how='left')
df_unique['patient_sex'] = df_unique['patient_sex'].fillna('unknown')
df_unique['patient_age_group'] = df_unique['patient_age_group'].fillna('unknown')

df_unique['strat_key'] = (
    df_unique['category'] + '|' + df_unique['patient_sex'] + '|' + df_unique['patient_age_group']
)

print("Category Distribution:")
print(df_unique['category'].value_counts())
print(f"Total unique images: {len(df_unique)}")

# Collapse strata with <4 members (can't survive two nested 80/10/10 splits)
strat_counts = df_unique['strat_key'].value_counts()
rare_keys = strat_counts[strat_counts < 4].index
df_unique['strat_key_final'] = np.where(
    df_unique['strat_key'].isin(rare_keys),
    df_unique['category'],
    df_unique['strat_key']
)
n_collapsed = df_unique['strat_key'].isin(rare_keys).sum()
print(f"Images with a rare label+sex+age stratum (collapsed to label-only): {n_collapsed}")

train_ids, temp_ids = train_test_split(
    df_unique, test_size=0.2, stratify=df_unique['strat_key_final'], random_state=42
)
val_ids, test_ids = train_test_split(
    temp_ids, test_size=0.5, stratify=temp_ids['strat_key_final'], random_state=42
)

print(f"\nTrain: {len(train_ids)} images")
print(f"Val:   {len(val_ids)} images")
print(f"Test:  {len(test_ids)} images")

train_ids[['image_id']].to_csv(f'{DATASET_DIR}/split_train.csv', index=False)
val_ids[['image_id']].to_csv(f'{DATASET_DIR}/split_val.csv', index=False)
test_ids[['image_id']].to_csv(f'{DATASET_DIR}/split_test.csv', index=False)

print("\nSplits saved")

## Reproducibility

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

## Dataset Class

In [ ]:

# Dataloaders function
def create_dataloaders(metadata_csv, images_dir, resolution, batch_size):

    train_dataset = ChestXrayDataset(
        metadata_csv, f'{DATASET_DIR}/split_train.csv', images_dir, resolution, is_for_train=True
    )
    val_dataset = ChestXrayDataset(
        metadata_csv, f'{DATASET_DIR}/split_val.csv', images_dir, resolution, is_for_train=False
    )
    test_dataset = ChestXrayDataset(
        metadata_csv, f'{DATASET_DIR}/split_test.csv', images_dir, resolution, is_for_train=False
    )

    # Multi-label-aware sampler weights (section 3.2): each image's weight
    # reflects the inverse frequency of its RAREST positive label.
    n_train = len(train_dataset.image_ids)
    label_positive_counts = {0: 0, 1: 0}
    image_labels = {}

    for image_id in train_dataset.image_ids:
        filas = train_dataset.df[train_dataset.df['image_id'] == image_id]
        positive_classes = [c for c in (0, 1) if c in filas['class_id'].values]
        image_labels[image_id] = positive_classes
        for c in positive_classes:
            label_positive_counts[c] += 1

    n_healthy = sum(1 for v in image_labels.values() if len(v) == 0)
    label_inv_freq = {
        c: (n_train / count if count > 0 else 0.0)
        for c, count in label_positive_counts.items()
    }
    healthy_weight = n_train / n_healthy if n_healthy > 0 else 0.0

    sample_weights = []
    for image_id in train_dataset.image_ids:
        positive_classes = image_labels[image_id]
        if not positive_classes:
            sample_weights.append(healthy_weight)
        else:
            sample_weights.append(max(label_inv_freq[c] for c in positive_classes))

    sampler = WeightedRandomSampler(
        weights=sample_weights, num_samples=len(train_dataset), replacement=True
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    print(f"Train: {len(train_dataset)} images")
    print(f"Val:   {len(val_dataset)} images")
    print(f"Test:  {len(test_dataset)} images")

    return train_loader, val_loader, test_loader

In [ ]:
def build_model(architecture, resolution, freeze_backbone=True):
    """
    architecture: 'alexnet' or 'densenet'
    resolution: 256 or 1024
    freeze_backbone: if True freezes everything except the last layer
    """
    if architecture == 'alexnet':
        model = models.alexnet(weights='IMAGENET1K_V1')
        if freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False
        # Substitute final classifier
        model.classifier[6] = nn.Linear(4096, 2)

        # Unfreeze last 3 conv layers + classifier for BOTH resolutions (section 2.1)
        for name, param in model.named_parameters():
            if any(f'features.{i}' in name for i in [8, 10, 12]):
                param.requires_grad = True

    elif architecture == 'densenet':
        model = models.densenet121(weights='IMAGENET1K_V1')
        if freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False
        # Substitute final classifier
        model.classifier = nn.Linear(1024, 2)

        # Unfreeze denseblock4 + classifier for BOTH resolutions (section 2.1)
        for name, param in model.named_parameters():
            if 'denseblock4' in name:
                param.requires_grad = True

    return model.to(DEVICE)


def train_model(model, train_loader, val_loader, architecture, resolution,
                    epochs=30, patience=5, pos_weight=None):

    lr = 1e-5  # equalized across resolutions (section 2.2)
    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight.to(DEVICE) if pos_weight is not None else None
    )
    scaler = GradScaler()  # Mixed precision

    best_val_loss = float('inf')
    epochs_without_improvement = 0
    record = {'train_loss': [], 'val_loss': []}

    for epoch in range(epochs):
        # Train
        model.train()
        train_loss = 0
        t0 = time.time()

        for imgs, labels, _ in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE
            )
            optimizer.zero_grad()

            with autocast(DEVICE):
                outputs = model(imgs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()

        train_loss /= len(train_loader)

        # Validation
        model.eval()
        val_loss = 0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for imgs, labels, _ in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                with autocast(DEVICE):
                    outputs = model(imgs)
                    loss = criterion(outputs, labels)
                val_loss += loss.item()
                all_preds.append((outputs.cpu().numpy() > 0.5).astype(int))
                all_labels.append(labels.cpu().numpy())

        val_loss /= len(val_loader)
        all_preds = np.concatenate(all_preds)
        all_labels = np.concatenate(all_labels)

        mcc_aneurysm = matthews_corrcoef(all_labels[:, 0], all_preds[:, 0])
        mcc_cardio = matthews_corrcoef(all_labels[:, 1], all_preds[:, 1])

        record['train_loss'].append(train_loss)
        record['val_loss'].append(val_loss)

        print(f"Epoch {epoch+1}/{epochs} | "
        f"Train: {train_loss:.4f} | Val: {val_loss:.4f} | "
        f"MCC [Aneu: {mcc_aneurysm:.3f} Card: {mcc_cardio:.3f}] | "
        f"Time: {time.time()-t0:.1f}s")

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(),
                      f'{MODELS_DIR}/{architecture}_{resolution}_best.pth')
            epochs_without_improvement = 0
            print(f"   ✓ Best model saved")
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"   Early stopping in epoch {epoch+1}")
                break

    return record


def compute_pos_weight(train_loader):
    """Per-label pos_weight = n_negative / n_positive for BCEWithLogitsLoss (section 3.3)."""
    ds = train_loader.dataset
    n_total = len(ds)
    n_pos = torch.zeros(2)
    for image_id in ds.image_ids:
        filas = ds.df[ds.df['image_id'] == image_id]
        if 0 in filas['class_id'].values:
            n_pos[0] += 1
        if 1 in filas['class_id'].values:
            n_pos[1] += 1
    n_neg = n_total - n_pos
    return n_neg / n_pos.clamp(min=1)

## AlexNet 256x256


In [ ]:
# Create dataloaders
train_loader, val_loader, test_loader = create_dataloaders(
    metadata_csv=f'{DATASET_DIR}/metadata_256.csv',
    images_dir=f'{DATASET_DIR}/images_256',
    resolution=256,
    batch_size=16
)

In [ ]:
# Build AlexNet
model = build_model('alexnet', resolution=256)

# Verify trainable parameters and freezed parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,}")

# Train and monitor AlexNet with 256x256 images
tracker = EmissionsTracker(output_dir=PROJECT_DIR,log_level="error")
tracker.start()

pos_weight = compute_pos_weight(train_loader)

record = train_model(model, train_loader, val_loader,
                        architecture='alexnet', resolution=256, epochs=50, pos_weight=pos_weight)

emissions = tracker.stop()
print(f"CO2 emissions: {emissions:.6f} kg")

## DenseNet-121 256x256

In [ ]:
train_loader, val_loader, test_loader = create_dataloaders(
    metadata_csv=f'{DATASET_DIR}/metadata_256.csv',
    images_dir=f'{DATASET_DIR}/images_256',
    resolution=256,
    batch_size=16
)

In [ ]:

model = build_model('densenet', resolution=256)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,}")

tracker = EmissionsTracker(output_dir=PROJECT_DIR,log_level="error")
tracker.start()

pos_weight = compute_pos_weight(train_loader)

record = train_model(model, train_loader, val_loader,
                        architecture='densenet', resolution=256, epochs=50, pos_weight=pos_weight)

emissions = tracker.stop()
print(f"CO2 emissions: {emissions:.6f} kg")

## AlexNet 1024x1024


In [ ]:
train_loader, val_loader, test_loader = create_dataloaders(
    metadata_csv=f'{DATASET_DIR}/metadata_1024.csv',
    images_dir=f'{DATASET_DIR}/images_1024',
    resolution=1024,
    batch_size=4
)

In [ ]:
model = build_model('alexnet', resolution=1024)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,}")

tracker = EmissionsTracker(output_dir=PROJECT_DIR,log_level="error")
tracker.start()

pos_weight = compute_pos_weight(train_loader)

record = train_model(model, train_loader, val_loader,
                        architecture='alexnet', resolution=1024,
                            epochs=50, patience=5, pos_weight=pos_weight)

emissions = tracker.stop()
print(f"CO2 emissions: {emissions:.6f} kg")

## DenseNet-121 1024x1024

In [ ]:
train_loader, val_loader, test_loader = create_dataloaders(
    metadata_csv=f'{DATASET_DIR}/metadata_1024.csv',
    images_dir=f'{DATASET_DIR}/images_1024',
    resolution=1024,
    batch_size=4
)

In [ ]:
model = build_model('densenet', resolution=1024)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,}")

tracker = EmissionsTracker(output_dir=PROJECT_DIR,log_level="error")
tracker.start()

pos_weight = compute_pos_weight(train_loader)

record = train_model(model, train_loader, val_loader,
                        architecture='densenet', resolution=1024,
                            epochs=50, patience=5, pos_weight=pos_weight)

emissions = tracker.stop()
print(f"CO2 emissions: {emissions:.6f} kg")

## Restore

In [ ]:
shutil.rmtree(f'{DATASET_DIR}')
print('Dataset folder deleted.')
print('02_preprocessing_images_and_training completed.')

# 03 - Models Clinical Validation

In [ ]:
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import (
    matthews_corrcoef, roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve, confusion_matrix
)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import shutil

import sys
sys.path.append('.')
from config import PROJECT_DIR, DATASET_ZIP, DATASET_DIR, MODELS_DIR, LABELS, MODELS, CONFIGS
from utils import build_model, ChestXrayDataset, apply_nclahe

In [ ]:
!unzip -q {DATASET_ZIP} -d {PROJECT_DIR}

## Reproducibility

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {DEVICE}')

In [ ]:
# Clinical evaluation function for each model

def clinical_eval_model(model, test_loader):
    model.eval()
    all_logits = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels, _ in test_loader:
            imgs = imgs.to(DEVICE)
            outputs = model(imgs)
            # Save logits (before sigmoid) for AUC
            all_logits.append(outputs.cpu().numpy())
            all_labels.append(labels.numpy())

    all_logits = np.concatenate(all_logits)     # (N, 2)
    all_labels = np.concatenate(all_labels)     # (N, 2)
    all_probs  = 1 / (1 + np.exp(-all_logits))  # sigmoid

    results = {}

    for i, name in enumerate(LABELS):
        y_true = all_labels[:, i]
        y_prob = all_probs[:, i]
    
        # Optimal threshold by MCC over ROC 
        fpr, tpr, thresholds = roc_curve(y_true, y_prob)
        mccs = []
        for t in thresholds:
            y_pred_t = (y_prob >= t).astype(int)
            mccs.append(matthews_corrcoef(y_true, y_pred_t))
        optimal_threshold = thresholds[np.argmax(mccs)]
        y_pred = (y_prob >= optimal_threshold).astype(int)

        # Clinical metrics
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        mcc = matthews_corrcoef(y_true, y_pred)
        auc_roc = roc_auc_score(y_true, y_prob)
        pr_auc  = average_precision_score(y_true, y_prob)

        threshold_max_sens = np.nan
        if name == 'aneurysm':
            true_pos_probs = y_prob[y_true == 1]
            if len(true_pos_probs) > 0:
                threshold_max_sens = float(true_pos_probs.min())

        results[name] = {
            'optimal threshold': round(optimal_threshold, 4),
            'threshold_max_sens': round(threshold_max_sens, 4) if not np.isnan(threshold_max_sens) else np.nan,
            'MCC':               round(mcc, 4),
            'AUC-ROC':           round(auc_roc, 4),
            'PR-AUC':            round(pr_auc, 4),
            'recall':            round(recall, 4),
            'specificity':       round(specificity, 4),
            'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn)
        }

    return results

In [ ]:
all_results = {}
global_thresholds = {}

for cfg in CONFIGS:

    print(f"\n{'='*50}")
    print(f"Evaluating {cfg['name']}...")

    # Dataset and loader
    test_dataset = ChestXrayDataset(
        metadata_csv=cfg['metadata'],
        split_csv=f"{DATASET_DIR}/split_test.csv",
        images_dir=cfg['images'],
        resolution=cfg['res']
    )
    test_loader = DataLoader(test_dataset, batch_size=cfg['batch'], shuffle=False)
    print(f"Test set: {len(test_dataset)} images")

    # Load model
    model = build_model(cfg['arq'], DEVICE)
    pth_path = f"{MODELS_DIR}/{cfg['arq']}_{cfg['res']}_best.pth"
    model.load_state_dict(torch.load(pth_path, map_location=DEVICE))
    print(f"Checkpoint loaded: {pth_path}")

    # Evaluate
    results = clinical_eval_model(model, test_loader)
    global_thresholds[cfg['name']] = {
    'aneurysm': results['aneurysm']['optimal threshold'],
    'cardiomegaly': results['cardiomegaly']['optimal threshold'],
    'aneurysm_max_sens': results['aneurysm']['threshold_max_sens'],
    }
    all_results[cfg['name']] = results

    # Print results
    for label, metrics in results.items():
        print(f"\n  {label}:")
        for k, v in metrics.items():
            print(f"    {k}: {v}")

print("\n✓ Evaluation completed.")

In [ ]:
# Comparative summary table
rows = []
for model, labels in all_results.items():
    for label, metrics in labels.items():
        rows.append({
            'Model': model,
            'Label': label,
            'MCC': metrics['MCC'],
            'AUC-ROC': metrics['AUC-ROC'],
            'PR-AUC': metrics['PR-AUC'],
            'Recall': metrics['recall'],
            'Specificity': metrics['specificity'],
            'Threshold': metrics['optimal threshold']
        })

df_results = pd.DataFrame(rows)
print(df_results.to_string(index=False))

# Save to PROJECT_DIR
df_results.to_csv(f'{PROJECT_DIR}/results_clinical_evaluation.csv', index=False)
print(f'\n✓ Saved: {PROJECT_DIR}/results_clinical_evaluation.csv')

In [ ]:
registers = []

for cfg in CONFIGS:
    test_dataset = ChestXrayDataset(
        metadata_csv=cfg['metadata'],
        split_csv=f"{DATASET_DIR}/split_test.csv",
        images_dir=cfg['images'],
        resolution=cfg['res']
    )
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

    model = build_model(cfg['arq'], DEVICE)
    pth_path = f"{MODELS_DIR}/{cfg['arq']}_{cfg['res']}_best.pth"
    model.load_state_dict(torch.load(pth_path, map_location=DEVICE))
    model.eval()

    u = global_thresholds[cfg['name']]
    df_demo = pd.read_csv(cfg['metadata']).drop_duplicates('image_id').set_index('image_id')

    with torch.no_grad():
        for imgs, labels, image_ids in test_loader:
            imgs = imgs.to(DEVICE)
            output = model(imgs)
            probs = torch.sigmoid(output).cpu().numpy()[0]
            label = labels.numpy()[0]
            image_id = image_ids[0]

            demo_row = df_demo.loc[image_id] if image_id in df_demo.index else None
            patient_age = demo_row['patient_age'] if demo_row is not None else np.nan
            patient_sex = demo_row['patient_sex'] if demo_row is not None else np.nan
            patient_age_group = demo_row['patient_age_group'] if demo_row is not None else np.nan

            registers.append({
                'model': cfg['name'],
                'image_id': image_id,
                'patient_age': patient_age,
                'patient_sex': patient_sex,
                'patient_age_group': patient_age_group,
                'prob_aneurysm': round(float(probs[0]), 4),
                'prob_cardiomegaly': round(float(probs[1]), 4),
                'pred_aneurysm': int(probs[0] >= u['aneurysm']),
                'pred_cardiomegaly': int(probs[1] >= u['cardiomegaly']),
                'label_aneurysm': int(label[0]),
                'label_cardiomegaly': int(label[1]),
                'threshold_mcc_aneurysm': u['aneurysm'],
                'threshold_mcc_cardiomegaly': u['cardiomegaly'],
                'threshold_max_sens_aneurysm': u['aneurysm_max_sens'],
                'pred_aneurysm_max_sens': int(probs[0] >= u['aneurysm_max_sens']) if not np.isnan(u['aneurysm_max_sens']) else np.nan,
            })

df_preds = pd.DataFrame(registers)
df_preds.to_csv(f"{PROJECT_DIR}/predictions_by_image.csv", index=False)
print(f'✓ Saved: {len(df_preds)} rows to predictions_by_image.csv')

## Approach to clinical valid threshold

In [ ]:
print("Minimum lower bound to have FN=0 in aortic enlargement by model:\n")

for model in ['AN_256', 'DN_256', 'AN_1024', 'DN_1024']:
    df_m = df_preds[df_preds['model'] == model].copy()

    # Only images with real aortic enlargement
    df_aneu = df_m[df_m['label_aneurysm'] == 1]

    # Minimum threshold = minimum probability among true positives
    threshold_fn0 = df_aneu['prob_aneurysm'].min()

    # With that threshold, calculate FP and specificity
    df_m['pred_new'] = (df_m['prob_aneurysm'] >= threshold_fn0).astype(int)

    TN = ((df_m['pred_new'] == 0) & (df_m['label_aneurysm'] == 0)).sum()
    FP = ((df_m['pred_new'] == 1) & (df_m['label_aneurysm'] == 0)).sum()
    TP = ((df_m['pred_new'] == 1) & (df_m['label_aneurysm'] == 1)).sum()
    FN = ((df_m['pred_new'] == 0) & (df_m['label_aneurysm'] == 1)).sum()

    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
    recall  = TP / (TP + FN) if (TP + FN) > 0 else 0

    print(f'{model}:')
    print(f'  Minimum threshold FN=0: {threshold_fn0:.4f}')
    print(f'  Recall: {recall:.4f} | Specificity: {specificity:.4f}')
    print(f'  TP: {TP} | FP: {FP} | TN: {TN} | FN: {FN}\n')

## Restore

In [ ]:
print(f'Removing {DATASET_DIR}...')
shutil.rmtree(DATASET_DIR)
print('Clinical validation of the models done')

# 04 - Saliency Maps

In [ ]:
import cv2
import numpy as np
import shutil
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models
import os
import zipfile

import sys
sys.path.append('.')
from config import PROJECT_DIR, MAPS_ZIP, MAPS_DIR, LABELS, DATASET_ZIP, DATASET_DIR, MODELS_DIR, CONFIGS
from utils import ChestXrayDataset, build_model

In [ ]:
os.system(f'unzip -q {DATASET_ZIP} -d {PROJECT_DIR}')

os.makedirs(f'{MAPS_DIR}', exist_ok=True)
print('.')

## Reproducibility

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
def get_target_layer(model, architecture):
    if architecture == 'alexnet':
        return model.features[12]
    elif architecture == 'densenet':
        return model.features.denseblock4

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(self._save_activations)
        target_layer.register_full_backward_hook(self._save_gradients)

    def _save_activations(self, module, input, output):
        self.activations = output.detach().clone()

    def _save_gradients(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach().clone()

    def generate(self, input_tensor, class_idx):
        input_tensor = input_tensor.requires_grad_(True)
        self.model.zero_grad()
        output = self.model(input_tensor)
        output[0, class_idx].backward(retain_graph=True)
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = torch.relu(cam.clone())
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

In [ ]:
# Verify resolution of saliency maps
for arq, res in [('alexnet', 256), ('alexnet', 1024), ('densenet', 256), ('densenet', 1024)]:
    if arq == 'alexnet':
        model = models.alexnet(weights=None)
        model.classifier[6] = nn.Linear(4096, 2)
        target = model.features[12]
    else:
        model = models.densenet121(weights=None)
        model.classifier = nn.Linear(1024, 2)
        target = model.features.norm5

    activation = {}
    def hook(module, input, output):
        activation['out'] = output

    target.register_forward_hook(hook)
    x = torch.zeros(1, 3, res, res)
    model(x)
    print(f"{arq} {res}x{res}: {activation['out'].shape}")

In [ ]:
for cfg in CONFIGS:
    name = cfg['name']
    print(f"\n{'='*50}")
    print(f"Generating maps for {name}...")

    out_dir = f'{MAPS_DIR}/{name}'
    os.makedirs(out_dir, exist_ok=True)

    dataset = ChestXrayDataset(
        metadata_csv=cfg['metadata'],
        split_csv=f'{DATASET_DIR}/split_test.csv',
        images_dir=cfg['images'],
        resolution=cfg['res']
    )
    loader = DataLoader(dataset, batch_size=1, shuffle=False)

    model = build_model(cfg['arq'], DEVICE)
    pth = f"{MODELS_DIR}/{cfg['arq']}_{cfg['res']}_best.pth"
    model.load_state_dict(torch.load(pth, map_location=DEVICE))
    model.eval()


    for module in model.modules():
        module._forward_hooks.clear()
        module._backward_hooks.clear()
        module._forward_pre_hooks.clear()

    target_layer = get_target_layer(model, cfg['arq'])
    gcam = GradCAM(model, target_layer=target_layer)

    for i, (img, label, image_id) in enumerate(loader):
        img = img.to(DEVICE)
        image_id = image_id[0]

        maps = {}
        for class_idx, label in enumerate(LABELS):
            cam =  gcam.generate(img, class_idx)
            res = cfg['res']
            # Bicubic upsampling to original resolution (section 8)
            maps[f'gradcam_{label}']   = cv2.resize(cam, (res, res), interpolation=cv2.INTER_CUBIC).astype(np.float32)
            maps[f'gradcam_{label}']   = np.clip(maps[f'gradcam_{label}'], 0.0, 1.0)

        np.savez_compressed(f'{out_dir}/{image_id}.npz', **maps)

        if (i + 1) % 100 == 0:
            print(f'  {i+1}/{len(dataset)} processed')

    print(f'  {name} completed.')

print('\nAll models completed.')

## Restore

In [ ]:
print('Compressing...')
with zipfile.ZipFile(f'{MAPS_ZIP}', 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(f'{MAPS_DIR}'):
        for file in files:
            filepath = os.path.join(root, file)
            arcname = os.path.relpath(filepath, f'{MAPS_DIR}')
            zf.write(filepath, arcname)

print('Done.')
print(f'Removing {DATASET_DIR} and {MAPS_DIR}...')
shutil.rmtree(DATASET_DIR)
shutil.rmtree(MAPS_DIR)

print('03_saliency_maps completed.')

# 05 - Maps Clinical Validation

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
from PIL import Image
from sklearn.metrics import auc
import shutil
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

import sys
sys.path.append('.')
from config import PROJECT_DIR, DATASET_DIR, DATASET_ZIP, MODELS_DIR, MAPS_DIR, MAPS_ZIP, CONFIGS, LABELS, LABEL_ID, IMAGENET_MEAN, IMAGENET_STD, MODELS
from utils import build_model, apply_nclahe, get_stratification, saliency_entropy, compute_mmd

In [ ]:
THRESHOLD_ACTIVATION = 0.2 # Threshold for binarizing activation maps when computing IoU.

In [ ]:
# Decompress dataset and saliency maps
print('Decompressing dataset and saliency maps...')
os.system(f'unzip -q {DATASET_ZIP} -d {PROJECT_DIR}')
os.system(f'unzip -q {MAPS_ZIP} -d {PROJECT_DIR}')
print('All done!')

## Reproducibility

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {DEVICE}')

In [ ]:
# Loading predictions by image
df_preds = pd.read_csv(f'{PROJECT_DIR}/predictions_by_image.csv')
print(f'Predictions loaded: {len(df_preds)} rows')
print(df_preds.head())

In [ ]:
# Saliency metrics
def pointing_game(cam, x_min, y_min, x_max, y_max):
    """1 if the maximum activation pixel falls within the BB, 0 otherwise."""
    max_idx = np.unravel_index(cam.argmax(), cam.shape)
    py, px = max_idx
    return int(x_min <= px <= x_max and y_min <= py <= y_max)


def poe(cam, x_min, y_min, x_max, y_max):
    """Proportion of energy of the map within the BB."""
    total = cam.sum()
    if total == 0:
        return 0.0
    x_min, y_min, x_max, y_max = int(x_min), int(y_min), int(x_max), int(y_max)
    dentro = cam[y_min:y_max, x_min:x_max].sum()
    return float(dentro / total)


# Heatmap/pathological anatomy convergence metrics - BB (requires binarization of the saliency map)

def binarize_map(map, threshold=0.20):
    """Binarizes the heatmap with a threshold over the maximum value."""
    return (map >= threshold * map.max()).astype(np.float32)


def dsc(cam_bin, x_min, y_min, x_max, y_max, res):
    """Dice Similarity Coefficient between binarized map and BB mask."""
    x_min, y_min, x_max, y_max = int(x_min), int(y_min), int(x_max), int(y_max)
    mask = np.zeros((res, res), dtype=np.float32)
    mask[y_min:y_max, x_min:x_max] = 1.0
    intersection = (cam_bin * mask).sum()
    suma = cam_bin.sum() + mask.sum()
    if suma == 0:
        return 0.0
    return float(2 * intersection / suma)


def iou_between_maps(cam1, cam2, threshold=0.20):
    """IoU between two binarized heatmaps (discrimination between labels)."""
    bin1 = binarize_map(cam1, threshold)
    bin2 = binarize_map(cam2, threshold)
    intersection = (bin1 * bin2).sum()
    union = ((bin1 + bin2) > 0).sum()
    if union == 0:
        return 0.0
    return float(intersection / union)

In [ ]:
# Deletion AUC and Insertion AUC

def deletion_insertion_auc(model, img_tensor, cam, class_idx, n_steps=10):
    """
    Calculates Deletion AUC and Insertion AUC.
     - Deletion: progressively removes the most important pixels
     - Insertion: progressively introduces the most important pixels
    Returns:
      (deletion_auc, insertion_auc)
    """
    model.eval()
    img_np = img_tensor.squeeze().cpu().numpy()  # (3, H, W)
    H, W = img_np.shape[1], img_np.shape[2]

    # Sort pixels by importance (highest to lowest)
    cam_flat = cam.flatten()
    sort = np.argsort(cam_flat)[::-1]

    # Reference image for insetion (blurred image)
    blur_img = cv2.GaussianBlur(
        img_np.transpose(1, 2, 0),
        (51, 51), 0
    ).transpose(2, 0, 1)

    deletion_scores = []
    insertion_scores = []

    n_pixels = H * W
    steps = [int(n_pixels * i / n_steps) for i in range(n_steps + 1)]

    img_del = img_np.copy().reshape(3, -1)
    img_ins = blur_img.copy().reshape(3, -1)

    for step in steps:
        # Deletion: put to zero the most important pixels
        img_del_step = img_np.copy().reshape(3, -1)
        img_del_step[:, sort[:step]] = 0
        tensor_del = torch.FloatTensor(img_del_step.reshape(3, H, W)).unsqueeze(0).to(DEVICE)

        # Insertion: reveal the most important pixels from the blurred image
        img_ins_step = blur_img.copy().reshape(3, -1)
        img_ins_step[:, sort[:step]] = img_np.reshape(3, -1)[:, sort[:step]]
        tensor_ins = torch.FloatTensor(img_ins_step.reshape(3, H, W)).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            prob_del = torch.sigmoid(model(tensor_del))[0, class_idx].item()
            prob_ins = torch.sigmoid(model(tensor_ins))[0, class_idx].item()

        deletion_scores.append(prob_del)
        insertion_scores.append(prob_ins)

    x = np.linspace(0, 1, n_steps + 1)
    del_auc = auc(x, deletion_scores)
    ins_auc = auc(x, insertion_scores)

    return del_auc, ins_auc

In [ ]:
class PenultimateFeatureExtractor:
    """Captures the penultimate-layer output for MMD (section 5.2).
    AlexNet: classifier[5] (ReLU before the final fc). DenseNet-121: the
    pooled feature vector feeding `classifier`, captured via a pre-hook."""
    def __init__(self, model, architecture):
        self.features = None
        self.architecture = architecture
        if architecture == 'alexnet':
            model.classifier[5].register_forward_hook(self._hook)
        elif architecture == 'densenet':
            model.classifier.register_forward_pre_hook(self._pre_hook)

    def _hook(self, module, input, output):
        self.features = output.detach().cpu().numpy()

    def _pre_hook(self, module, input):
        self.features = input[0].detach().cpu().numpy()

    def get(self):
        return self.features.squeeze(0)

In [ ]:
# Main

df_agg_metrics = pd.read_csv(f'{PROJECT_DIR}/results_clinical_evaluation.csv')

registers = []
features_by_group = {}  # {(model, label): {'TP': [...], 'FP': [...], 'FN': [...], 'TN': [...], 'pathology': [...], 'control': [...]}}

for cfg in CONFIGS:
    name = cfg['name']
    res = cfg['res']
    print(f"\n{'='*50}")
    print(f"Calculating metrics for {name}...")

    df_meta_full = pd.read_csv(cfg['metadata'])
    df_meta = df_meta_full[df_meta_full['class_id'].isin([0, 1])]
    df_pred_model = df_preds[df_preds['model'] == name]

    model = build_model(cfg['arq'], DEVICE)
    pth = f"{MODELS_DIR}/{cfg['arq']}_{res}_best.pth"
    model.load_state_dict(torch.load(pth, map_location=DEVICE))
    model.eval()
    for module in model.modules():
        if isinstance(module, torch.nn.ReLU):
            module.inplace = False

    feature_extractor = PenultimateFeatureExtractor(model, cfg['arq'])

    normalize = transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    maps_dir = f'{MAPS_DIR}/{name}'
    image_ids = df_pred_model['image_id'].unique()

    for i, image_id in enumerate(image_ids):
        npz_path = f'{maps_dir}/{image_id}.npz'
        if not os.path.exists(npz_path):
            continue
        maps = np.load(npz_path)

        img_path = f'{DATASET_DIR}/images_256/{image_id}.png' if res == 256 else f'{DATASET_DIR}/images_1024/{image_id}.png'
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            print(f'Not found: {img_path}')
            continue

        tile_size = 4 if res == 256 else 16
        img_clahe = apply_nclahe(img, tile_size=tile_size)
        img_rgb = cv2.cvtColor(img_clahe, cv2.COLOR_GRAY2RGB)
        img_tensor = normalize(transforms.ToTensor()(Image.fromarray(img_rgb))).unsqueeze(0)

        pred_row = df_pred_model[df_pred_model['image_id'] == image_id]
        if len(pred_row) == 0:
            continue
        pred_row = pred_row.iloc[0]

        with torch.no_grad():
            model(img_tensor.to(DEVICE))
        features_vec = feature_extractor.get()

        is_control = (pred_row['label_aneurysm'] == 0) and (pred_row['label_cardiomegaly'] == 0)

        for label_name in LABELS:
            class_idx = LABEL_ID[label_name]
            cam_key = f'gradcam_{label_name}'
            if cam_key not in maps:
                continue
            cam = maps[cam_key]

            label_col, pred_col = f'label_{label_name}', f'pred_{label_name}'
            label = int(pred_row[label_col])
            pred = int(pred_row[pred_col])

            if label == 1 and pred == 1: stratification = 'TP'
            elif label == 0 and pred == 1: stratification = 'FP'
            elif label == 1 and pred == 0: stratification = 'FN'
            else: stratification = 'TN'

            bb_rows = df_meta[(df_meta['image_id'] == image_id) & (df_meta['class_id'] == class_idx)]
            has_bb = len(bb_rows) > 0
            pg = poe_val = dsc_val = np.nan
            if has_bb:
                bb = bb_rows.iloc[0]
                pg = pointing_game(cam, bb['x_min'], bb['y_min'], bb['x_max'], bb['y_max'])
                poe_val = poe(cam, bb['x_min'], bb['y_min'], bb['x_max'], bb['y_max'])
                cam_bin = binarize_map(cam)
                dsc_val = dsc(cam_bin, bb['x_min'], bb['y_min'], bb['x_max'], bb['y_max'], res)

            iou_maps = np.nan
            cam_other_key = 'gradcam_cardiomegaly' if label_name == 'aneurysm' else 'gradcam_aneurysm'
            if cam_other_key in maps.files:
                iou_maps = iou_between_maps(cam, maps[cam_other_key])

            del_auc, ins_auc = deletion_insertion_auc(model, img_tensor, cam, class_idx, n_steps=10)
            entropy_val = saliency_entropy(cam)

            group_key = (name, label_name)
            features_by_group.setdefault(group_key, {
                'TP': [], 'FP': [], 'FN': [], 'TN': [], 'pathology': [], 'control': [],
                'by_sex': {}, 'by_age': {},
            })
            features_by_group[group_key][stratification].append(features_vec)
            if label == 1:
                features_by_group[group_key]['pathology'].append(features_vec)
                patient_sex = pred_row.get('patient_sex', np.nan)
                patient_age_group = pred_row.get('patient_age_group', np.nan)
                if pd.notna(patient_sex):
                    features_by_group[group_key]['by_sex'].setdefault(patient_sex, []).append(features_vec)
                if pd.notna(patient_age_group) and patient_age_group != 'unknown':
                    features_by_group[group_key]['by_age'].setdefault(patient_age_group, []).append(features_vec)
            if is_control:
                features_by_group[group_key]['control'].append(features_vec)

            agg_row = df_agg_metrics[(df_agg_metrics['Model'] == name) & (df_agg_metrics['Label'] == label_name)]
            mcc_agg = agg_row['MCC'].values[0] if len(agg_row) else np.nan
            auc_roc_agg = agg_row['AUC-ROC'].values[0] if len(agg_row) else np.nan
            pr_auc_agg = agg_row['PR-AUC'].values[0] if len(agg_row) else np.nan
            sens_agg = agg_row['Recall'].values[0] if len(agg_row) else np.nan
            spec_agg = agg_row['Specificity'].values[0] if len(agg_row) else np.nan

            threshold_mcc = pred_row.get(f'threshold_mcc_{label_name}', np.nan)
            threshold_max_sens = pred_row.get('threshold_max_sens_aneurysm', np.nan) if label_name == 'aneurysm' else np.nan

            registers.append({
                'image_id': image_id, 'model': name, 'label': label_name, 'resolution': res, 'iteracion': 1,
                'patient_age': pred_row.get('patient_age', np.nan),
                'patient_age_group': pred_row.get('patient_age_group', np.nan),
                'patient_sex': pred_row.get('patient_sex', np.nan),
                'verdict': stratification,
                'threshold_mcc': threshold_mcc, 'threshold_max_sens': threshold_max_sens,
                'MCC': mcc_agg, 'AUC_ROC': auc_roc_agg, 'PR_AUC': pr_auc_agg,
                'sensitivity': sens_agg, 'specificity': spec_agg,
                'PG': pg, 'PoE': poe_val, 'DSC': dsc_val, 'IoU_between_maps': iou_maps,
                'Del_AUC': del_auc, 'Ins_AUC': ins_auc, 'entropy_gradcam': entropy_val,
            })

        if (i + 1) % 100 == 0:
            print(f'  {i+1}/{len(image_ids)} processed')

    print(f'  {name} completed.')

df_metrics = pd.DataFrame(registers)
print(f'\n Computed per-image-per-label rows: {len(df_metrics)}')

In [ ]:
MIN_GROUP_SIZE = 2
mmd_results = {}

def max_pairwise_mmd(groups_by_key):
    """Max MMD across all pairs of demographic strata (section 5.2): a large
    value flags that a pathology's latent representation is conditioned on
    the demographic attribute, i.e. a demographic-shortcut signature."""
    keys = [k for k, v in groups_by_key.items() if len(v) >= MIN_GROUP_SIZE]
    pair_values = {}
    for i in range(len(keys)):
        for j in range(i + 1, len(keys)):
            k1, k2 = keys[i], keys[j]
            pair_values[f'{k1}_vs_{k2}'] = compute_mmd(
                np.array(groups_by_key[k1]), np.array(groups_by_key[k2])
            )
    if not pair_values:
        return np.nan, {}
    return max(pair_values.values()), pair_values

for (model_name, label_name), groups in features_by_group.items():
    result = {'pathology_vs_control': np.nan, 'TP_vs_FP': np.nan, 'TP_vs_FN': np.nan}
    if len(groups['pathology']) >= MIN_GROUP_SIZE and len(groups['control']) >= MIN_GROUP_SIZE:
        result['pathology_vs_control'] = compute_mmd(np.array(groups['pathology']), np.array(groups['control']))
    if len(groups['TP']) >= MIN_GROUP_SIZE and len(groups['FP']) >= MIN_GROUP_SIZE:
        result['TP_vs_FP'] = compute_mmd(np.array(groups['TP']), np.array(groups['FP']))
    if len(groups['TP']) >= MIN_GROUP_SIZE and len(groups['FN']) >= MIN_GROUP_SIZE:
        result['TP_vs_FN'] = compute_mmd(np.array(groups['TP']), np.array(groups['FN']))

    result['MMD_sex'], sex_pairs = max_pairwise_mmd(groups['by_sex'])
    result['MMD_age'], age_pairs = max_pairwise_mmd(groups['by_age'])
    if sex_pairs or age_pairs:
        print(f"{model_name}/{label_name} demographic MMD (pathology-positive only):")
        for k, v in {**sex_pairs, **age_pairs}.items():
            print(f"    {k}: {v:.4f}")

    mmd_results[(model_name, label_name)] = result

mmd_cross_label = {}
for model_name in MODELS:
    feats_aneu = features_by_group.get((model_name, 'aneurysm'), {}).get('pathology', [])
    feats_card = features_by_group.get((model_name, 'cardiomegaly'), {}).get('pathology', [])
    if len(feats_aneu) >= MIN_GROUP_SIZE and len(feats_card) >= MIN_GROUP_SIZE:
        mmd_cross_label[model_name] = compute_mmd(np.array(feats_aneu), np.array(feats_card))
    else:
        mmd_cross_label[model_name] = np.nan

print("MMD cardiomegaly vs aortic enlargement, by model:")
for model_name, v in mmd_cross_label.items():
    print(f"  {model_name}: {v}")

df_metrics['MMD_pathology_vs_control'] = df_metrics.apply(
    lambda r: mmd_results.get((r['model'], r['label']), {}).get('pathology_vs_control', np.nan), axis=1)
df_metrics['MMD_TP_vs_FP'] = df_metrics.apply(
    lambda r: mmd_results.get((r['model'], r['label']), {}).get('TP_vs_FP', np.nan), axis=1)
df_metrics['MMD_TP_vs_FN'] = df_metrics.apply(
    lambda r: mmd_results.get((r['model'], r['label']), {}).get('TP_vs_FN', np.nan), axis=1)
df_metrics['MMD_sex'] = df_metrics.apply(
    lambda r: mmd_results.get((r['model'], r['label']), {}).get('MMD_sex', np.nan), axis=1)
df_metrics['MMD_age'] = df_metrics.apply(
    lambda r: mmd_results.get((r['model'], r['label']), {}).get('MMD_age', np.nan), axis=1)

UNIFIED_COLUMNS = [
    'image_id', 'model', 'label', 'resolution', 'iteracion',
    'patient_age', 'patient_age_group', 'patient_sex',
    'verdict', 'threshold_mcc', 'threshold_max_sens',
    'MCC', 'AUC_ROC', 'PR_AUC', 'sensitivity', 'specificity',
    'PG', 'PoE', 'DSC', 'IoU_between_maps', 'Del_AUC', 'Ins_AUC',
    'entropy_gradcam', 'MMD_pathology_vs_control', 'MMD_TP_vs_FP', 'MMD_TP_vs_FN',
    'MMD_sex', 'MMD_age',
]
df_metrics = df_metrics[UNIFIED_COLUMNS]
df_metrics.to_csv(f'{PROJECT_DIR}/xai_metrics.csv', index=False)
print(f"\nSaved unified results: {len(df_metrics)} rows to xai_metrics.csv")

In [ ]:
df = pd.read_csv(f'{PROJECT_DIR}/xai_metrics.csv')

print('\n=== Global metrics (per model, per label) ===')
global_agg = df.groupby(['model', 'label']).agg(
    PG=('PG', 'mean'), PoE=('PoE', 'mean'), DSC=('DSC', 'mean'),
    IoU_between_maps=('IoU_between_maps', 'mean'),
    Del_AUC=('Del_AUC', 'mean'), Ins_AUC=('Ins_AUC', 'mean'),
    entropy_gradcam=('entropy_gradcam', 'mean'), n=('image_id', 'count')
).round(4)
print(global_agg.to_string())

print('\n=== Stratified by verdict (TP/FP/FN) ===')
verdict_agg = df[df['verdict'] != 'TN'].groupby(['model', 'label', 'verdict']).agg(
    PG=('PG', 'mean'), PoE=('PoE', 'mean'), DSC=('DSC', 'mean'),
    Del_AUC=('Del_AUC', 'mean'), Ins_AUC=('Ins_AUC', 'mean'),
    entropy_gradcam=('entropy_gradcam', 'mean'), n=('image_id', 'count')
).round(4)
print(verdict_agg.to_string())

print('\n=== Stratified by PatientSex ===')
sex_agg = df.groupby(['model', 'label', 'patient_sex']).agg(
    PG=('PG', 'mean'), PoE=('PoE', 'mean'), DSC=('DSC', 'mean'),
    Del_AUC=('Del_AUC', 'mean'), Ins_AUC=('Ins_AUC', 'mean'),
    entropy_gradcam=('entropy_gradcam', 'mean'), n=('image_id', 'count')
).round(4)
print(sex_agg.to_string())

print('\n=== Stratified by PatientAge group ===')
age_agg = df.groupby(['model', 'label', 'patient_age_group']).agg(
    PG=('PG', 'mean'), PoE=('PoE', 'mean'), DSC=('DSC', 'mean'),
    Del_AUC=('Del_AUC', 'mean'), Ins_AUC=('Ins_AUC', 'mean'),
    entropy_gradcam=('entropy_gradcam', 'mean'), n=('image_id', 'count')
).round(4)
print(age_agg.to_string())

print('\nReminder: entropy_gradcam comparisons must stay within-architecture (AN vs AN, DN vs DN).')

global_agg.to_csv(f'{PROJECT_DIR}/global_xai_summary.csv')
verdict_agg.to_csv(f'{PROJECT_DIR}/stratification_verdict_xai_summary.csv')
sex_agg.to_csv(f'{PROJECT_DIR}/stratification_sex_xai_summary.csv')
age_agg.to_csv(f'{PROJECT_DIR}/stratification_age_xai_summary.csv')
print('\nTables saved')

In [ ]:
# Full analytics breakdown: model -> label -> verdict -> sex -> age group, with
# counts. This is the exhaustive table for analysis; the four tables above are
# the ones surfaced in the paper.
full_agg = df.groupby(['model', 'label', 'verdict', 'patient_sex', 'patient_age_group']).agg(
    PG=('PG', 'mean'), PoE=('PoE', 'mean'), DSC=('DSC', 'mean'),
    IoU_between_maps=('IoU_between_maps', 'mean'),
    Del_AUC=('Del_AUC', 'mean'), Ins_AUC=('Ins_AUC', 'mean'),
    entropy_gradcam=('entropy_gradcam', 'mean'),
    MMD_pathology_vs_control=('MMD_pathology_vs_control', 'mean'),
    MMD_TP_vs_FP=('MMD_TP_vs_FP', 'mean'),
    MMD_TP_vs_FN=('MMD_TP_vs_FN', 'mean'),
    MMD_sex=('MMD_sex', 'mean'),
    MMD_age=('MMD_age', 'mean'),
    n=('image_id', 'count'),
).round(4)
print(f'\n=== Full breakdown: model x label x verdict x sex x age ({len(full_agg)} rows) ===')
print(full_agg.to_string())

full_agg.to_csv(f'{PROJECT_DIR}/full_stratified_xai_summary.csv')
print('\nFull table saved to full_stratified_xai_summary.csv')

In [ ]:
## IoU between maps of both labels

In [ ]:
df_preds = pd.read_csv(f'{PROJECT_DIR}/predictions_by_image.csv')

df_preds['type_aneurysm']     = df_preds.apply(lambda r: get_stratification(r, 'aneurysm'), axis=1)
df_preds['type_cardiomegaly'] = df_preds.apply(lambda r: get_stratification(r, 'cardiomegaly'), axis=1)
df_preds['combo']              = df_preds['type_aneurysm'] + '/' + df_preds['type_cardiomegaly']

results = []

for model in MODELS:
    print(f'Processing {model}...')
    df_m = df_preds[df_preds['model'] == model]

    for _, row in df_m.iterrows():
        image_id = row['image_id']
        combo    = row['combo']

        path = os.path.join(MAPS_DIR, model, f'{image_id}.npz')
        if not os.path.exists(path):
            continue

        data    = np.load(path)
        mapa_an = data['gradcam_aneurysm']
        mapa_ca = data['gradcam_cardiomegaly']

        iou = iou_between_maps(mapa_an, mapa_ca)

        results.append({
            'model':   model,
            'image_id': image_id,
            'combo':    combo,
            'iou':      iou
        })

df_iou = pd.DataFrame(results)

table = df_iou.groupby(['model', 'combo'])['iou'].agg(['mean', 'count']).round(4)
table.columns = ['IoU medio', 'n']
print('\nAverage IoU between maps by model and verdict combo:\n')
print(table.to_string())

df_iou.to_csv(f'{PROJECT_DIR}/iou_stratified_combo.csv', index=False)
table.to_csv(f'{PROJECT_DIR}/iou_stratified_summary.csv', index=False)
print('\nSaved.')

## Restore

In [ ]:
print(f'Removing {DATASET_DIR} and {MAPS_DIR}...')
shutil.rmtree(DATASET_DIR)
shutil.rmtree(MAPS_DIR)